# Fine-Tuning Ministral-3B for Crisis Social Media Response Generation

**Ripple** is a multi-agent crisis simulation engine that generates realistic media content during corporate crises. This notebook fine-tunes [Ministral-3B-Instruct](https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512) (3.4B params, Apache 2.0) using **LoRA** on the pre-quantized FP8 model to produce Reddit/social-media-style crisis responses.

- **Runtime**: Kaggle/Colab free tier (T4 GPU), ~15-25 min training
- **Dataset**: ~800 programmatically generated examples (no API calls needed)
- **Output**: LoRA adapter (~20-50MB vs 6.8GB full model)

| Component | Detail |
|-----------|--------|
| Base model | `mistralai/Ministral-3-3B-Instruct-2512` |
| Quantization | FP8 (pre-quantized by Mistral) |
| LoRA rank | 16, alpha 32 |
| Training | 3 epochs, lr=2e-4, cosine schedule |
| Framework | PEFT + TRL SFTTrainer |

In [ ]:
# Cell 1: Install Dependencies
# Using PEFT + BitsAndBytes for QLoRA fine-tuning (no Unsloth — avoids compiler bugs with Ministral-3)

!pip install -q --upgrade "transformers>=4.46.0" "trl>=0.12.0" \
    "peft>=0.14.0" "bitsandbytes>=0.45.0" "accelerate>=1.2.0" \
    "mistral-common>=1.5.0" datasets matplotlib

In [ ]:
# Cell 2: Generate Synthetic Training Dataset
# Programmatically generates ~800 crisis response examples using templates + randomization.
# No LLM API calls needed — fully reproducible.

import random
import json
from collections import Counter

random.seed(42)

# ─── Crisis Scenarios (15 scenarios) ───────────────────────────────────────────
CRISIS_SCENARIOS = [
    {
        "name": "data_breach",
        "context": "A major data breach at TechVault Inc has exposed 12 million customer records including emails, passwords, and partial credit card numbers. The breach was discovered by a security researcher who found the data on a dark web forum.",
        "company": "TechVault Inc",
        "cohorts": ["General Public", "Tech Community", "Investors", "Regulators", "Employees"],
    },
    {
        "name": "ceo_fraud",
        "context": "FinCore's CEO has been charged with securities fraud after an SEC investigation revealed systematic earnings manipulation over 3 years. Stock price dropped 40% in pre-market trading.",
        "company": "FinCore",
        "cohorts": ["Investors", "Employees", "General Public", "Media", "Regulators"],
    },
    {
        "name": "product_recall",
        "context": "SafeHome Corp is recalling 2.3 million smart smoke detectors after reports of battery fires in 47 homes. Three injuries have been reported. The CPSC has issued an urgent safety warning.",
        "company": "SafeHome Corp",
        "cohorts": ["General Public", "Tech Community", "Investors", "Regulators", "Media"],
    },
    {
        "name": "oil_spill",
        "context": "PetroGlobal's offshore drilling platform has leaked an estimated 50,000 barrels of crude oil into the Gulf coast. Environmental groups are mobilizing and satellite imagery shows a 12-mile oil slick.",
        "company": "PetroGlobal",
        "cohorts": ["Environmental Groups", "General Public", "Investors", "Regulators", "Local Communities"],
    },
    {
        "name": "ai_bias",
        "context": "FairLend AI's loan approval algorithm has been found to systematically deny applications from minority neighborhoods at 3x the rate of comparable white neighborhoods. A ProPublica investigation has gone viral.",
        "company": "FairLend AI",
        "cohorts": ["Civil Rights Groups", "Tech Community", "General Public", "Regulators", "Investors"],
    },
    {
        "name": "ransomware",
        "context": "MedNet Healthcare's hospital network has been hit by ransomware, shutting down systems at 14 hospitals across 5 states. Emergency patients are being diverted. Hackers demand $15M in Bitcoin.",
        "company": "MedNet Healthcare",
        "cohorts": ["Patients", "Healthcare Workers", "General Public", "Regulators", "Tech Community"],
    },
    {
        "name": "factory_explosion",
        "context": "An explosion at ChemWorks' Texas plant has killed 4 workers and injured 22 others. Toxic fumes forced evacuation of a 3-mile radius. OSHA had cited the plant for safety violations twice in the past year.",
        "company": "ChemWorks",
        "cohorts": ["Local Communities", "Employees", "Regulators", "General Public", "Investors"],
    },
    {
        "name": "social_media_scandal",
        "context": "Internal documents leaked from BuzzSphere show the company knowingly amplified misinformation to boost engagement metrics. A former VP has turned whistleblower and is testifying before Congress.",
        "company": "BuzzSphere",
        "cohorts": ["General Public", "Advertisers", "Tech Community", "Regulators", "Employees"],
    },
    {
        "name": "food_contamination",
        "context": "FreshFarms' organic spinach has been linked to an E. coli outbreak affecting 230 people across 8 states, with 12 hospitalizations. The FDA has traced contamination to a single processing facility.",
        "company": "FreshFarms",
        "cohorts": ["General Public", "Health Community", "Investors", "Regulators", "Retailers"],
    },
    {
        "name": "labor_abuse",
        "context": "An undercover documentary reveals systematic labor exploitation at GigDash's warehouse network — workers forced to skip bathroom breaks, 14-hour shifts, and retaliation against union organizers.",
        "company": "GigDash",
        "cohorts": ["Workers", "General Public", "Labor Unions", "Investors", "Regulators"],
    },
    {
        "name": "privacy_scandal",
        "context": "SmartWatch Pro has been secretly recording user conversations and selling transcripts to advertisers. A class-action lawsuit has been filed representing 5 million device owners.",
        "company": "SmartWatch Pro",
        "cohorts": ["General Public", "Tech Community", "Privacy Advocates", "Regulators", "Investors"],
    },
    {
        "name": "crypto_collapse",
        "context": "CryptoNova exchange has frozen all withdrawals after a $800M shortfall was discovered. The CEO has gone silent and the company's headquarters appears abandoned. Users fear a rug pull.",
        "company": "CryptoNova",
        "cohorts": ["Crypto Community", "Investors", "General Public", "Regulators", "Media"],
    },
    {
        "name": "autonomous_vehicle_crash",
        "context": "DriveAI's autonomous taxi struck and killed a pedestrian in San Francisco. Dashcam footage shows the vehicle failed to brake. This is the third incident in 6 months involving DriveAI vehicles.",
        "company": "DriveAI",
        "cohorts": ["General Public", "Tech Community", "Regulators", "Investors", "Transportation Workers"],
    },
    {
        "name": "pharma_cover_up",
        "context": "Internal emails show PharmaCure knew about severe side effects of its top-selling painkiller Relivex but suppressed clinical trial data. The drug has been prescribed to 8 million patients.",
        "company": "PharmaCure",
        "cohorts": ["Patients", "Healthcare Workers", "General Public", "Regulators", "Investors"],
    },
    {
        "name": "airline_safety",
        "context": "SkyLink Airlines flight SL-447 made an emergency landing after a door panel blew off at 30,000 feet. Maintenance logs reveal the airline has been deferring critical inspections to cut costs.",
        "company": "SkyLink Airlines",
        "cohorts": ["Passengers", "Aviation Workers", "General Public", "Regulators", "Investors"],
    },
]

# ─── Speaker Archetypes by Message Type ────────────────────────────────────────
SPEAKER_ARCHETYPES = {
    "forum": [
        {"role": "Affected Customer", "personality": "angry, frustrated, sharing personal experience", "reach_range": (0.05, 0.3)},
        {"role": "Concerned Parent", "personality": "worried, protective, seeking information", "reach_range": (0.05, 0.25)},
        {"role": "Industry Insider (Anonymous)", "personality": "cautious, revealing, insider knowledge", "reach_range": (0.1, 0.4)},
        {"role": "Skeptical Observer", "personality": "questioning, analytical, doubts official narrative", "reach_range": (0.05, 0.3)},
        {"role": "Long-time Customer", "personality": "disappointed, loyal but shaken, comparing past experience", "reach_range": (0.05, 0.2)},
        {"role": "Community Moderator", "personality": "balanced, trying to organize information, fact-checking", "reach_range": (0.1, 0.35)},
        {"role": "Former Employee", "personality": "bitter, revealing internal culture problems", "reach_range": (0.1, 0.4)},
        {"role": "Local Resident", "personality": "scared, directly impacted, emotional", "reach_range": (0.05, 0.2)},
    ],
    "news": [
        {"role": "Senior Reporter at Reuters", "personality": "factual, authoritative, breaking news style", "reach_range": (0.7, 0.95)},
        {"role": "Investigative Journalist at ProPublica", "personality": "persistent, deep-dive, connecting dots", "reach_range": (0.5, 0.8)},
        {"role": "Tech Reporter at The Verge", "personality": "tech-savvy, analytical, industry context", "reach_range": (0.5, 0.75)},
        {"role": "Business Correspondent at CNBC", "personality": "market-focused, financial impact, shareholder perspective", "reach_range": (0.6, 0.85)},
        {"role": "Local News Reporter", "personality": "community-focused, human interest, on-the-ground", "reach_range": (0.2, 0.45)},
        {"role": "Columnist at Washington Post", "personality": "opinion-driven, broader societal implications", "reach_range": (0.55, 0.8)},
    ],
    "influencer": [
        {"role": "Tech YouTuber (450K subscribers)", "personality": "outraged, dramatic, calls for accountability", "reach_range": (0.4, 0.7)},
        {"role": "Finance Twitter personality", "personality": "sarcastic, meme-heavy, market commentary", "reach_range": (0.3, 0.6)},
        {"role": "Consumer Rights Advocate", "personality": "passionate, organized, rallying followers", "reach_range": (0.3, 0.55)},
        {"role": "Industry Analyst on LinkedIn", "personality": "measured, professional, systemic analysis", "reach_range": (0.3, 0.6)},
        {"role": "Activist with large following", "personality": "confrontational, demanding action, hashtag campaigns", "reach_range": (0.4, 0.7)},
        {"role": "Popular Podcaster", "personality": "conversational, speculative, audience engagement", "reach_range": (0.35, 0.65)},
        {"role": "Meme Account Operator", "personality": "satirical, viral, gallows humor", "reach_range": (0.2, 0.5)},
    ],
    "official": [
        {"role": "Company PR Spokesperson", "personality": "measured, corporate language, damage control", "reach_range": (0.5, 0.8)},
        {"role": "Government Regulator", "personality": "formal, authoritative, announcing investigations", "reach_range": (0.6, 0.9)},
        {"role": "Company CEO", "personality": "apologetic, taking responsibility, promising action", "reach_range": (0.7, 0.95)},
        {"role": "Industry Trade Group", "personality": "diplomatic, industry-wide perspective, self-regulatory", "reach_range": (0.3, 0.6)},
    ],
}

# ─── Name Pool ─────────────────────────────────────────────────────────────────
FIRST_NAMES = [
    "Marcus", "Priya", "Jake", "Aisha", "Carlos", "Emma", "Wei", "Sarah",
    "Dmitri", "Kenji", "Fatima", "Lucas", "Amara", "Ryan", "Zara", "Tyler",
    "Mei", "Jordan", "Kayla", "Omar", "Sofia", "David", "Nina", "Alex",
    "Chen", "Isabella", "Jamal", "Elena", "Raj", "Mika", "Dante", "Yuki",
]
LAST_NAMES = [
    "Webb", "Chen", "Rodriguez", "Okafor", "Kim", "Patel", "Nguyen", "Murphy",
    "Kowalski", "Tanaka", "Hassan", "Silva", "Williams", "Zhang", "Johansson",
    "Reeves", "Bakshi", "Torres", "Andersen", "Nakamura", "Foster", "Moreau",
]

def random_name():
    return f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"

def random_handle(name):
    styles = [
        lambda n: f"@{n.split()[0].lower()}_{n.split()[1].lower()}",
        lambda n: f"@{n.split()[0].lower()}{n.split()[1].lower()}",
        lambda n: f"@Real{n.split()[0]}",
        lambda n: f"@{n.split()[0].lower()}{random.randint(88, 99)}",
    ]
    return random.choice(styles)(name)

# ─── Message Templates by Type and Sentiment Phase ─────────────────────────────

FORUM_TEMPLATES = {
    "escalation": [
        "Has anyone else heard about what's going on with {company}? I just saw something about {event_hint} and I'm honestly shocked. How is this even possible in {year}?",
        "OK so I've been a {company} customer for {years} years and I just found out about {event_hint}. I am LIVID. How do they get away with this? Anyone else considering switching?",
        "Thread: Everything we know about the {company} situation so far. I've been following this since it broke and here's what I've gathered... {event_hint}. This is way bigger than they're admitting.",
        "My {relation} works at {company} and they're saying internally it's way worse than what's being reported. Apparently {event_hint} and management knew for months.",
        "Just got an email from {company} about {event_hint}. The absolute NERVE of their PR team to send such a generic response. They don't care about us.",
        "I'm literally shaking. I trusted {company} with my {trust_item} and now {event_hint}? This is a complete betrayal of their customers.",
        "Can someone explain to me how {company} thought {event_hint} was acceptable? I've been reading the reports and each one is worse than the last.",
        "PSA: If you're a {company} customer, you need to {action} RIGHT NOW. {event_hint} means your {trust_item} could be compromised.",
        "The fact that {company} is still operating after {event_hint} tells you everything about how broken the system is. No accountability whatsoever.",
        "I work in {industry} and I can tell you — {event_hint} at {company} is just the tip of the iceberg. This is an industry-wide problem.",
    ],
    "pivot": [
        "{company} just released their official response and... it's {response_quality}. {response_reaction}. I'm not sure if this is enough.",
        "So {company}'s CEO finally spoke up about {event_hint}. Honestly {response_reaction}. We'll see if they follow through.",
        "Update on the {company} situation: they're now {action}. {response_reaction}. I want to believe they're serious but their track record...",
        "After {company}'s announcement today, I'm cautiously {sentiment_word}. {response_reaction}. Actions speak louder than words though.",
        "Interesting development — {company} is bringing in {external_party} to handle {event_hint}. {response_reaction}.",
    ],
    "fatigue": [
        "Another day, another update on {company}. At this point I'm just exhausted following this story. Wake me up when someone actually faces consequences.",
        "Is anyone else just... tired of the {company} saga? It's been {days} days and nothing meaningful has changed.",
        "Honest question — does anyone still care about the {company} thing? I feel like we've all moved on but nothing was actually resolved.",
        "The {company} situation is exactly what's wrong with {industry}. Same pattern every time: crisis, outrage, PR response, everyone forgets. See you at the next one.",
        "Unpopular opinion: the {company} crisis is being overblown at this point. Yes it was bad, but the initial response was {response_quality} and they're working on it.",
    ],
}

NEWS_TEMPLATES = {
    "escalation": [
        "BREAKING: {company} faces {event_type} as {event_hint}. Sources confirm the situation is \"rapidly evolving.\" Full coverage to follow.",
        "EXCLUSIVE: Documents obtained by our newsroom reveal {company} was aware of {event_hint} as early as {months_ago} months ago. The company declined to comment.",
        "{company} stock plummets {pct}% in early trading as {event_hint} sends shockwaves through {industry}. Analysts warn of further downside risk.",
        "Federal regulators announce investigation into {company} following {event_hint}. \"We are taking this matter very seriously,\" said a spokesperson.",
        "Victims of {company}'s {event_type} speak out: \"We trusted them and they failed us.\" Our reporters talked to {num_affected} affected {stakeholders}.",
        "{company} crisis deepens: New reports reveal {event_hint}. Industry experts say this could be one of the worst {event_type} incidents in recent memory.",
    ],
    "pivot": [
        "{company} CEO addresses {event_type} in first public statement: \"{ceo_quote}.\" Market reaction is {market_reaction}.",
        "ANALYSIS: {company}'s response to {event_type} — what worked, what didn't, and what comes next. Our {industry} desk breaks it down.",
        "{company} announces {action} in response to {event_type}. Experts are {response_quality_adj} about the plan's effectiveness.",
    ],
    "fatigue": [
        "Week {week} of {company} crisis: Public attention wanes but affected {stakeholders} say their fight is far from over.",
        "One month later: Where does {company} stand after {event_type}? A look at what's changed — and what hasn't.",
        "As {company} crisis fades from headlines, regulators quietly continue probe. Settlement talks reportedly underway.",
    ],
}

INFLUENCER_TEMPLATES = {
    "escalation": [
        "I cannot BELIEVE what {company} just did. {event_hint} is absolutely unacceptable. Thread incoming 🧵 (1/{thread_count})",
        "Just made a video breaking down the {company} situation. TL;DR: it's worse than you think. {event_hint} and they KNEW about it.",
        "The {company} {event_type} is exactly why I've been warning about {industry} for years. When will people listen? #{company}Crisis",
        "Y'all remember when I said {company} was sus? Well... {event_hint}. I hate being right about this stuff.",
        "Ok but can we talk about how {company} literally {event_hint} and nobody in mainstream media is covering the full story? Let me fill you in.",
        "Dropping everything to cover the {company} crisis. This is MASSIVE. {event_hint}. If you have {trust_item} with them, you need to act NOW.",
        "NEW VIDEO: How {company} destroyed the trust of {num_affected} {stakeholders} and what it means for the entire {industry}. Link in bio.",
    ],
    "pivot": [
        "{company}'s response just dropped and honestly? {response_reaction}. I'll do a full breakdown on my channel tonight.",
        "So {company} thinks {action} will fix things? {response_reaction}. The bar is on the floor and they're still tripping over it.",
        "UPDATE on {company}: They're finally {action}. This is {response_quality}. I'll give credit where it's due — IF they follow through.",
        "Hot take: {company}'s new plan is actually {response_quality}. I know everyone wants to hate on them but {response_reaction}.",
    ],
    "fatigue": [
        "Still monitoring the {company} situation but engagement has dropped 70% on my posts about it. People are over it. The algorithm has moved on.",
        "Final {company} update unless something major happens: {summary}. I'm moving on to cover {next_topic}.",
        "The {company} thing perfectly illustrates the attention span of social media. Massive outrage for a week, then... nothing. The affected people are still suffering.",
    ],
}

OFFICIAL_TEMPLATES = {
    "escalation": [
        "We are aware of the situation and are conducting a thorough investigation. The safety and trust of our {stakeholders} remains our top priority. We will provide updates as more information becomes available.",
        "OFFICIAL STATEMENT: {company} takes these reports extremely seriously. We have engaged {external_party} to conduct an independent review. We are cooperating fully with all relevant authorities.",
        "To our valued {stakeholders}: We understand your concerns regarding {event_hint}. We want to assure you that we are taking immediate steps to address this situation.",
    ],
    "pivot": [
        "Today we are announcing a comprehensive action plan to address {event_type}. Key steps include: {action}. We are committed to earning back your trust through actions, not just words.",
        "Following our thorough review, we are implementing the following changes effective immediately: {action}. We have also appointed {external_party} to oversee compliance.",
        "An open letter from our CEO: \"I take full responsibility for {event_hint}. {ceo_quote}. We owe you better, and we will deliver better.\"",
    ],
    "fatigue": [
        "Progress update on our remediation efforts: {action}. We remain committed to full transparency and will continue providing regular updates to all stakeholders.",
        "As we enter the next phase of our response to {event_type}, we want to share the progress made so far and outline our continued commitment to {stakeholders}.",
    ],
}

# ─── Fill-in values ────────────────────────────────────────────────────────────

FILL_VALUES = {
    "year": ["2024", "2025", "2026"],
    "years": ["3", "5", "7", "10", "12"],
    "relation": ["friend", "cousin", "neighbor", "college roommate", "partner"],
    "trust_item": ["personal data", "money", "health information", "safety", "private photos", "financial records"],
    "action": [
        "hiring a new Chief Safety Officer", "launching an independent audit",
        "offering full refunds to all affected customers", "shutting down the affected systems",
        "issuing a voluntary recall", "partnering with external investigators",
        "implementing new safety protocols", "setting up a victim compensation fund",
        "firing the responsible executives", "releasing all internal documents",
    ],
    "external_party": [
        "Deloitte", "a former federal prosecutor", "an independent safety board",
        "a top-tier cybersecurity firm", "PricewaterhouseCoopers", "a court-appointed monitor",
    ],
    "response_quality": ["not great", "underwhelming", "surprisingly decent", "better than expected", "terrible", "actually pretty good"],
    "response_quality_adj": ["cautiously optimistic", "skeptical", "divided", "hopeful but wary"],
    "response_reaction": [
        "too little too late if you ask me", "I actually think they're taking this seriously",
        "classic corporate damage control", "better than nothing I suppose",
        "I want to believe them but I've been burned before",
        "the affected people deserve better than vague promises",
        "at least they're acknowledging the problem now",
    ],
    "sentiment_word": ["optimistic", "hopeful", "skeptical", "neutral"],
    "industry": ["tech", "finance", "healthcare", "energy", "consumer goods", "social media", "transportation"],
    "event_type": ["crisis", "scandal", "safety incident", "data breach", "fraud allegations", "regulatory probe"],
    "pct": ["8", "12", "15", "22", "30", "40"],
    "months_ago": ["3", "6", "9", "12", "18"],
    "num_affected": ["dozens of", "hundreds of", "thousands of", "over a million"],
    "stakeholders": ["customers", "users", "patients", "employees", "investors", "community members"],
    "market_reaction": ["mixed", "cautiously positive", "negative", "sharply negative"],
    "ceo_quote": [
        "We failed our customers and we own that failure completely",
        "I am personally committed to making this right",
        "What happened is inexcusable and I take full responsibility",
        "We will not rest until every affected person has been made whole",
    ],
    "thread_count": ["8", "12", "15", "20"],
    "summary": [
        "company is slowly addressing things but there's still a long way to go",
        "the regulatory investigation is ongoing and could take months",
        "affected users are still waiting for meaningful compensation",
    ],
    "next_topic": ["the new AI regulations", "another breaking tech story", "something more positive"],
    "week": ["2", "3", "4"],
}

def fill_template(template: str, scenario: dict) -> str:
    """Fill a template with scenario-specific and random values."""
    text = template.replace("{company}", scenario["company"])
    # Create event hints from the scenario context
    context_phrases = scenario["context"].split(". ")
    event_hint = random.choice(context_phrases).strip().rstrip(".")
    text = text.replace("{event_hint}", event_hint.lower() if len(event_hint) > 60 else event_hint)
    # Fill remaining placeholders
    for key, values in FILL_VALUES.items():
        placeholder = "{" + key + "}"
        while placeholder in text:
            text = text.replace(placeholder, random.choice(values), 1)
    return text


def get_narrative_phase(day: int) -> str:
    """Map simulation day to narrative arc phase."""
    if day <= 3:
        return "escalation"
    elif day <= 7:
        return "pivot"
    else:
        return "fatigue"


def sentiment_for_phase(phase: str, msg_type: str) -> float:
    """Generate a sentiment value appropriate for the narrative phase and message type."""
    ranges = {
        "escalation": {"forum": (-0.9, -0.3), "news": (-0.6, -0.1), "influencer": (-0.95, -0.2), "official": (-0.3, 0.1)},
        "pivot": {"forum": (-0.5, 0.4), "news": (-0.3, 0.3), "influencer": (-0.6, 0.5), "official": (0.0, 0.5)},
        "fatigue": {"forum": (-0.4, 0.2), "news": (-0.2, 0.1), "influencer": (-0.3, 0.3), "official": (0.1, 0.6)},
    }
    lo, hi = ranges[phase].get(msg_type, (-0.5, 0.5))
    return round(random.uniform(lo, hi), 2)


# ─── System Prompt (aligned with Ripple's tick-orchestrator.ts) ─────────────────
SYSTEM_PROMPT = """You are a crisis simulation engine generating realistic social media and news content.
Generate messages as specific speaker characters reacting to a corporate crisis.
Each message should feel authentic to its platform:
- forum: Reddit/forum style — personal, emotional, community discussion
- news: Professional journalism — factual, sourced, structured
- influencer: Social media personality — dramatic, engaging, platform-native
- official: Corporate/government communications — measured, formal, strategic

Messages must reflect the speaker's personality and the crisis timeline:
- ESCALATION (days 1-3): Shock, anger, breaking news, uncertainty
- PIVOT (days 4-7): Company response, divided reactions, analysis
- FATIGUE (days 8-14): Waning attention, resolution, lasting impacts

Content should feel like real posts from real people — not generic AI text."""


# ─── Generate the dataset ──────────────────────────────────────────────────────

TEMPLATE_MAP = {
    "forum": FORUM_TEMPLATES,
    "news": NEWS_TEMPLATES,
    "influencer": INFLUENCER_TEMPLATES,
    "official": OFFICIAL_TEMPLATES,
}

# Target distribution: forum 300, news 200, influencer 200, official 100
TYPE_TARGETS = {"forum": 300, "news": 200, "influencer": 200, "official": 100}
PERIODS = ["Morning", "Afternoon", "Evening"]

dataset = []
type_counts = Counter()

for msg_type, target in TYPE_TARGETS.items():
    templates_by_phase = TEMPLATE_MAP[msg_type]
    archetypes = SPEAKER_ARCHETYPES[msg_type]

    for i in range(target):
        scenario = random.choice(CRISIS_SCENARIOS)
        day = random.randint(1, 14)
        phase = get_narrative_phase(day)
        period = random.choice(PERIODS)
        tick_index = PERIODS.index(period)

        archetype = random.choice(archetypes)
        name = random_name()
        handle = random_handle(name)
        reach = round(random.uniform(*archetype["reach_range"]), 2)
        sentiment = sentiment_for_phase(phase, msg_type)

        # Pick a template and fill it
        templates = templates_by_phase[phase]
        template = random.choice(templates)
        content = fill_template(template, scenario)

        # Build the user prompt (crisis context)
        cohort = random.choice(scenario["cohorts"])
        user_prompt = (
            f"Crisis: {scenario['context']}\n"
            f"Day {day}, {period} tick (tickIndex={tick_index})\n"
            f"Speaker: {handle} ({name}) — {archetype['role']} in {cohort}\n"
            f"Personality: {archetype['personality']}\n"
            f"Message type: {msg_type}, reach: {reach}, sentiment: {sentiment}\n"
            f"Narrative phase: {phase.upper()}\n"
            f"Generate a single realistic {msg_type} message from this speaker."
        )

        # Build the ChatML conversation
        example = {
            "conversations": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": content},
            ],
            # Metadata for analysis
            "metadata": {
                "msg_type": msg_type,
                "scenario": scenario["name"],
                "day": day,
                "phase": phase,
                "period": period,
                "sentiment": sentiment,
                "reach": reach,
                "cohort": cohort,
            },
        }
        dataset.append(example)
        type_counts[msg_type] += 1

random.shuffle(dataset)

print(f"Generated {len(dataset)} training examples")
print(f"Distribution: {dict(type_counts)}")
print(f"Scenarios covered: {len(set(ex['metadata']['scenario'] for ex in dataset))}")
print(f"\nSample conversation:")
print(json.dumps(dataset[0]["conversations"], indent=2)[:1500])

In [ ]:
# Cell 3: Dataset Visualization

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# ─── 1. Message Type Distribution ──────────────────────────────────────────────
types = [ex["metadata"]["msg_type"] for ex in dataset]
type_counter = Counter(types)
colors_types = ["#4C78A8", "#F58518", "#E45756", "#72B7B2"]
bars = axes[0].bar(type_counter.keys(), type_counter.values(), color=colors_types, edgecolor="white", linewidth=0.5)
axes[0].set_title("Message Type Distribution", fontweight="bold", fontsize=11)
axes[0].set_ylabel("Count")
for bar, count in zip(bars, type_counter.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(count),
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# ─── 2. Sentiment Distribution ─────────────────────────────────────────────────
sentiments = [ex["metadata"]["sentiment"] for ex in dataset]
axes[1].hist(sentiments, bins=30, color="#54A24B", edgecolor="white", linewidth=0.5, alpha=0.85)
axes[1].axvline(x=0, color="#333", linestyle="--", linewidth=0.8, alpha=0.5)
axes[1].set_title("Sentiment Distribution", fontweight="bold", fontsize=11)
axes[1].set_xlabel("Sentiment (-1 to 1)")
axes[1].set_ylabel("Count")
mean_sent = sum(sentiments) / len(sentiments)
axes[1].annotate(f'mean={mean_sent:.2f}', xy=(mean_sent, 0), xytext=(mean_sent + 0.3, max(plt.gca().get_ylim()) * 0.8),
                arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')

# ─── 3. Crisis Scenario Coverage ───────────────────────────────────────────────
scenarios = [ex["metadata"]["scenario"] for ex in dataset]
scenario_counter = Counter(scenarios)
sorted_scenarios = sorted(scenario_counter.items(), key=lambda x: x[1], reverse=True)
labels = [s[0].replace("_", " ").title() for s in sorted_scenarios]
values = [s[1] for s in sorted_scenarios]
axes[2].barh(labels, values, color="#B279A2", edgecolor="white", linewidth=0.5)
axes[2].set_title("Crisis Scenario Coverage", fontweight="bold", fontsize=11)
axes[2].set_xlabel("Count")
axes[2].invert_yaxis()
for i, v in enumerate(values):
    axes[2].text(v + 0.5, i, str(v), va='center', fontsize=8)

plt.tight_layout()
plt.savefig("dataset_distribution.png", bbox_inches="tight", facecolor="white")
plt.show()

# ─── Print Example Conversations ───────────────────────────────────────────────
print("\n" + "="*80)
print("EXAMPLE TRAINING CONVERSATIONS")
print("="*80)

# Show one of each type
shown_types = set()
for ex in dataset:
    mt = ex["metadata"]["msg_type"]
    if mt not in shown_types:
        shown_types.add(mt)
        meta = ex["metadata"]
        print(f"\n{'─'*80}")
        print(f"Type: {mt.upper()} | Scenario: {meta['scenario']} | Day {meta['day']} ({meta['phase']}) | Sentiment: {meta['sentiment']}")
        print(f"{'─'*80}")
        for msg in ex["conversations"]:
            role = msg["role"].upper()
            content_preview = msg["content"][:300] + ("..." if len(msg["content"]) > 300 else "")
            print(f"[{role}]: {content_preview}")
    if len(shown_types) >= 4:
        break

In [ ]:
# Cell 4: Load Model with LoRA
# Ministral-3B-Instruct-2512 is pre-quantized in FP8 (~3.4GB VRAM)
# We load it as-is and apply LoRA adapters on top

import torch
from transformers import Mistral3ForConditionalGeneration, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "mistralai/Ministral-3-3B-Instruct-2512"
MAX_SEQ_LENGTH = 2048

# Load the FP8 pre-quantized model directly
model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    attn_implementation="eager",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# Prepare for training
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Apply LoRA adapters on the language model layers
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Print parameter stats
model.print_trainable_parameters()
print(f"Memory footprint: ~{torch.cuda.memory_allocated() / 1e9:.1f} GB VRAM")

In [ ]:
# Cell 5: Train with SFTTrainer

from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import matplotlib.pyplot as plt

# Pre-format all examples into text using the tokenizer's chat template.
formatted_texts = []
for ex in dataset:
    text = tokenizer.apply_chat_template(
        ex["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    formatted_texts.append(text)

train_data = Dataset.from_dict({"text": formatted_texts})

print(f"Training dataset: {len(train_data)} examples")
print(f"Sample (first 300 chars): {train_data[0]['text'][:300]}...")

# Training configuration
sft_config = SFTConfig(
    dataset_text_field="text",
    output_dir="./crisis-model-output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 16
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    max_length=MAX_SEQ_LENGTH,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    save_strategy="no",  # We'll save manually at the end
    optim="adamw_8bit",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    args=sft_config,
)

print(f"\nStarting training...")
print(f"  Epochs: {sft_config.num_train_epochs}")
print(f"  Batch size: {sft_config.per_device_train_batch_size} x {sft_config.gradient_accumulation_steps} = {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Learning rate: {sft_config.learning_rate}")
print(f"  Scheduler: {sft_config.lr_scheduler_type}")

train_result = trainer.train()

# ─── Plot Training Loss ────────────────────────────────────────────────────────
log_history = trainer.state.log_history
train_losses = [(entry["step"], entry["loss"]) for entry in log_history if "loss" in entry]

if train_losses:
    steps, losses = zip(*train_losses)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, losses, color="#E45756", linewidth=1.5, alpha=0.8)
    ax.fill_between(steps, losses, alpha=0.1, color="#E45756")
    ax.set_xlabel("Training Step", fontsize=11)
    ax.set_ylabel("Loss", fontsize=11)
    ax.set_title("Training Loss — Ministral-3B Crisis Fine-Tuning", fontweight="bold", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig("training_loss.png", bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"\nFinal loss: {losses[-1]:.4f}")
    print(f"Loss reduction: {losses[0]:.4f} → {losses[-1]:.4f} ({(1 - losses[-1]/losses[0])*100:.1f}% decrease)")

print(f"\nTraining complete!")
print(f"  Total steps: {trainer.state.global_step}")
print(f"  Training time: {train_result.metrics.get('train_runtime', 0):.0f}s")

In [ ]:
# Cell 6: Inference Demo
# Test with 5 scenarios NOT in the training data to evaluate generalization

model.eval()

TEST_SCENARIOS = [
    {
        "context": "CloudStack's entire platform has been down for 18 hours, affecting 200,000 businesses worldwide. The company has not issued any public statement.",
        "speaker": "@dev_sarah_k (Sarah Kim) — Frustrated SaaS Developer in Tech Community",
        "personality": "technical, sarcastic, sharing workarounds",
        "msg_type": "forum",
        "day": 1, "period": "Afternoon", "phase": "ESCALATION",
        "sentiment": -0.7, "reach": 0.2,
    },
    {
        "context": "GreenEnergy Corp's wind turbines have been found to contain toxic chemicals leaking into farmland soil across 3 states. EPA launches emergency investigation.",
        "speaker": "@ReutersEnergy (Reuters Energy Desk) — Senior Energy Correspondent",
        "personality": "factual, authoritative, breaking news style",
        "msg_type": "news",
        "day": 2, "period": "Morning", "phase": "ESCALATION",
        "sentiment": -0.4, "reach": 0.85,
    },
    {
        "context": "EduTech Academy's online tutoring platform exposed children's private conversations and location data to third-party advertisers.",
        "speaker": "@parentwatch_mike (Mike Torres) — Child Safety Advocate with 230K followers",
        "personality": "outraged, protective, rallying parents",
        "msg_type": "influencer",
        "day": 3, "period": "Evening", "phase": "ESCALATION",
        "sentiment": -0.9, "reach": 0.55,
    },
    {
        "context": "AutoPilot Motors' self-driving trucks caused a multi-vehicle pileup on I-95. After 5 days of silence, the company announces a full fleet recall and independent safety review.",
        "speaker": "@AutoPilotMotors (Official Account) — VP of Communications",
        "personality": "measured, corporate language, damage control",
        "msg_type": "official",
        "day": 5, "period": "Evening", "phase": "PIVOT",
        "sentiment": 0.1, "reach": 0.8,
    },
    {
        "context": "NutriFit's protein supplements were found to contain undisclosed steroids. After 10 days, public interest is fading despite ongoing FDA investigation.",
        "speaker": "@gym_life_jason99 (Jason Andersen) — Fitness Community Regular",
        "personality": "tired, cynical, seen-it-all attitude",
        "msg_type": "forum",
        "day": 11, "period": "Morning", "phase": "FATIGUE",
        "sentiment": -0.15, "reach": 0.1,
    },
]

print("="*80)
print("INFERENCE DEMO — Fine-Tuned Crisis Response Generation")
print("="*80)

for i, scenario in enumerate(TEST_SCENARIOS):
    user_prompt = (
        f"Crisis: {scenario['context']}\n"
        f"Day {scenario['day']}, {scenario['period']} tick (tickIndex={['Morning', 'Afternoon', 'Evening'].index(scenario['period'])})\n"
        f"Speaker: {scenario['speaker']}\n"
        f"Personality: {scenario['personality']}\n"
        f"Message type: {scenario['msg_type']}, reach: {scenario['reach']}, sentiment: {scenario['sentiment']}\n"
        f"Narrative phase: {scenario['phase']}\n"
        f"Generate a single realistic {scenario['msg_type']} message from this speaker."
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    # Decode only the generated part
    generated = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    print(f"\n{'─'*80}")
    print(f"Test {i+1}: {scenario['msg_type'].upper()} | Day {scenario['day']} ({scenario['phase']}) | Sentiment: {scenario['sentiment']}")
    print(f"Speaker: {scenario['speaker']}")
    print(f"Crisis: {scenario['context'][:100]}...")
    print(f"{'─'*80}")
    print(f"\n{generated.strip()}\n")

In [ ]:
# Cell 7: Save LoRA Adapter
# Only saves the fine-tuned adapter weights (~20-50MB) instead of the full model (~6.8GB)

import os

ADAPTER_PATH = "./crisis-lora-adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

# Calculate adapter size
adapter_size = sum(
    os.path.getsize(os.path.join(dirpath, f))
    for dirpath, _, filenames in os.walk(ADAPTER_PATH)
    for f in filenames
)

full_model_size_gb = 6.8  # Ministral-3B full FP16 size
adapter_size_mb = adapter_size / (1024 * 1024)

print(f"LoRA adapter saved to: {ADAPTER_PATH}/")
print(f"")
print(f"Size comparison:")
print(f"  Full model:   {full_model_size_gb:.1f} GB")
print(f"  LoRA adapter: {adapter_size_mb:.1f} MB")
print(f"  Compression:  {full_model_size_gb * 1024 / adapter_size_mb:.0f}x smaller")
print(f"")
print(f"Files saved:")
for f in sorted(os.listdir(ADAPTER_PATH)):
    fpath = os.path.join(ADAPTER_PATH, f)
    fsize = os.path.getsize(fpath)
    if fsize > 1024 * 1024:
        print(f"  {f}: {fsize / (1024*1024):.1f} MB")
    else:
        print(f"  {f}: {fsize / 1024:.1f} KB")

print(f"\nTo reload this adapter later:")
print(f"  from peft import PeftModel")
print(f"  model = AutoModelForCausalLM.from_pretrained('{MODEL_NAME}', ...)")
print(f"  model = PeftModel.from_pretrained(model, '{ADAPTER_PATH}')")